
# LangGraph ReAct 에이전트와 메모리(MemorySaver)
## ReAct 에이전트란? 사용하면 무엇이 좋은가
**개념**
- **ReAct** = **Re**asoning(추론) + **Act**ing(행동). LLM 이 "무엇이 필요한지 추론 → 도구 호출(행동) → 결과 관찰 → 다시 추론"을 **답할 수 있을 때까지 반복**하는 에이전트 패턴입니다.
- 도구를 쓸지, 어떤 도구를 쓸지, 몇 번 쓸지를 코드가 아니라 **LLM 이 매 단계 스스로 결정**합니다.

**사용하면 얻는 효과**
- **유연한 질문 처리**: 메뉴 질문은 `search_menu`, 최신 정보는 `search_web` 처럼 질문에 맞는 도구를 골라 쓰고, 도구가 필요 없으면 바로 답합니다.
- **다단계 문제 해결**: 한 번의 검색으로 부족하면 결과를 보고 추가로 검색하거나, 여러 도구를 조합해 답합니다. (고정된 파이프라인에서는 어려움)
- **환각 감소**: 모르는 내용을 지어내는 대신 도구로 근거를 가져와 답합니다.
- **확장 용이**: 도구 함수만 추가하면 기능이 늘어납니다. 분기 로직을 직접 작성할 필요가 없습니다.
- 한계: LLM 호출이 반복되므로 **지연·비용이 늘 수 있고**, 도구 선택이 항상 정확하지는 않습니다.
### ReAct 루프
```text
START → agent ──(tool_calls 있음)──→ tools ──┐   행동(Act): 도구 호출
          ↑                                   │   관찰(Observe): 도구 결과를 모델에 전달
          └───────────────────────────────────┘   추론(Reason): 다음 행동 또는 최종 답변 결정
          └──(tool_calls 없음)──→ END
```
## MemorySaver 란? 사용하면 무엇이 좋은가
**개념**
- **MemorySaver** 는 그래프 실행의 **상태(state)를 단계마다 메모리에 저장**하는 체크포인터(checkpointer)입니다. 컴파일(또는 `create_agent`) 때 `checkpointer=MemorySaver()` 로 연결합니다.
- 저장된 상태는 **`thread_id`** 별로 구분됩니다. 같은 `thread_id` 로 호출하면 이전 대화 내용을 이어받습니다.

**사용하면 얻는 효과**
- **대화 기억**: LLM 자체는 호출 사이에 기억이 없지만, 체크포인트가 이전 메시지를 복원해 "아까 말한 메뉴 가격은?" 같은 **후속 질문**이 가능합니다. (멀티턴 대화)
- **대화 분리**: `thread_id` 가 다르면 서로 다른 대화로 격리되어, 사용자·세션별 대화가 섞이지 않습니다.
- **상태 확인·재개**: `get_state` 로 저장된 상태를 조회하고, 중단 지점부터 이어서 실행할 수 있습니다.
- 한계: `MemorySaver` 는 **프로세스 메모리에만 저장**하므로 재시작하면 사라집니다. 영구 보관은 `SqliteSaver`, `PostgresSaver` 등을 사용합니다.



## 이 노트북의 역할
레스토랑 **메뉴 검색(벡터DB)** 과 **웹 검색(Tavily)** 두 도구를 가진 ReAct 에이전트를 만들고, 체크포인터(`MemorySaver`)로 **대화를 기억**하게 한 뒤 Gradio 챗봇으로 연결합니다.

| 단계 | 내용 | 배우는 것 |
|---|---|---|
| 1. 환경 설정 | API 키, `get_llm()`, 라이브러리 | 제공자 선택 |
| 2. Tool 정의 | `search_menu`(FAISS), `search_web`(Tavily) | 실전 도구 작성, LLM 의 도구 선택 확인 |
| 3. ReAct Agent | `create_agent` / `StateGraph` + `tools_condition` | ReAct 루프, 두 가지 구성 방법 |
| 4. MemorySaver | 체크포인터, `thread_id` | 멀티턴 대화 기억, 대화 스레드 분리 |
| 5. Gradio 챗봇 | 웹 UI 연결 | 세션별 `thread_id` |


## 사용 모델 (`get_llm()`)
- `PROVIDER` 값(`"groq"` / `"openai"` / `"upstage"`, 기본값 `"upstage"`)만 바꾸면 이후 모든 셀의 LLM 이 바뀝니다. 바꾼 뒤에는 LLM 설정 셀부터 다시 실행하세요.
- **임베딩은 항상 Upstage** (`solar-embedding-1-large`) 입니다. `../db/menu_db` 가 이 모델로 만들어져 있어 다른 임베딩으로 바꾸면 검색이 되지 않습니다. 따라서 어떤 제공자를 골라도 `UPSTAGE_API_KEY` 는 필요합니다.
- 웹 검색 도구에는 `TAVILY_API_KEY` 가 필요합니다.
- **사전 준비**: 메뉴 벡터DB(`../db/menu_db`)는 `01_LangGraph_StateGraph` 노트북이 만들어 저장합니다. 이 폴더가 없으면 01 노트북을 먼저 실행하세요.
- LLM·임베딩·웹 검색 호출은 사용 요금이 발생할 수 있습니다.

## 참고
* 도구(`@tool`), `bind_tools`, `ToolNode` 의 기본 원리와 `StateGraph` 로 에이전트 루프를 직접 구성하는 방법은 **`03_LangGraph_Tool활용Basic`** 에서 다룹니다. 이 노트북에서는 반복하지 않습니다.
* 예전의 `langgraph.prebuilt.create_react_agent` 는 deprecated 이므로 `langchain.agents.create_agent` 를 사용합니다.

## 1. 환경 설정

`(1) API 키`

In [ ]:
from dotenv import load_dotenv
import os

# .env 파일을 불러와서 환경 변수로 설정
load_dotenv()

# 제공자별 API 키 (키 값은 출력하지 않음)
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
UPSTAGE_API_KEY = os.getenv("UPSTAGE_API_KEY")  # 임베딩에 항상 필요

# Tavily 는 환경변수 TAVILY_API_KEY 를 라이브러리가 직접 읽음 — 설정 여부만 확인
print("TAVILY_API_KEY 설정됨:", bool(os.getenv("TAVILY_API_KEY")))

`(3) LLM 설정 (Groq / OpenAI / Upstage 선택)`
- `PROVIDER` 값만 바꾸면 이후 모든 셀의 모델이 바뀝니다. (Groq 는 OpenAI 호환 API 라서 `ChatOpenAI` 에 `base_url` 만 다르게 연결)
- 바꾼 뒤에는 이 셀부터 다시 실행하세요. OpenAI 선택 시 사용 요금이 발생합니다.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_upstage import ChatUpstage

# 사용할 LLM 제공자 선택: "groq", "openai", "upstage" — 이 값만 바꾸면 이후 모든 셀의 모델이 바뀜
PROVIDER = "upstage"

# 제공자별 설정 (cls: 사용할 LangChain 채팅 모델 클래스)
LLM_CONFIGS = {
    "groq": {
        "cls": ChatOpenAI,  # Groq 는 OpenAI 호환 API 라서 base_url 만 다름
        "api_key": GROQ_API_KEY,
        "base_url": "https://api.groq.com/openai/v1",  # Groq API 엔드포인트
        "model": "openai/gpt-oss-120b",
    },
    "openai": {
        "cls": ChatOpenAI,
        "api_key": OPENAI_API_KEY,
        "base_url": None,  # 생략 시 OpenAI 기본 엔드포인트 사용
        "model": "gpt-4o-mini",  # 테스트에는 작은 모델 사용 (대안: "gpt-4o")
    },
    "upstage": {
        "cls": ChatUpstage,
        "api_key": UPSTAGE_API_KEY,
        "base_url": "https://api.upstage.ai/v1",
        "model": "solar-pro3",
    },
}
MAX_RETRIES = 5  # 503 등 일시 장애 자동 재시도 횟수


def get_llm(temperature: float = 0.5, provider: str | None = None, verbose: bool = True, **kwargs):
    """선택한 제공자(기본값: PROVIDER)의 채팅 모델 인스턴스를 생성합니다.
    verbose=True 이면 생성된 모델 정보를 출력합니다.
    kwargs: 모델 클래스에 그대로 전달할 추가 옵션 (예: seed=42, timeout=60)"""
    provider = provider or PROVIDER
    if provider not in LLM_CONFIGS:
        raise ValueError(f"지원하지 않는 제공자: {provider} (사용 가능: {list(LLM_CONFIGS)})")
    config = dict(LLM_CONFIGS[provider])
    model_cls = config.pop("cls")
    if not config["api_key"]:
        raise RuntimeError(f".env 파일에 {provider.upper()}_API_KEY 가 없습니다.")
    llm = model_cls(**config, temperature=temperature, max_retries=MAX_RETRIES, **kwargs)
    if verbose:
        # 생성된 모델 정보 출력 (API 키는 출력하지 않음)
        print(f"[LLM 생성] provider={provider} | model={llm.model_name}")
    return llm

In [ ]:
# 도구 호출·에이전트는 결과가 일정해야 하므로 temperature=0 사용
llm = get_llm(temperature=0)

`(2) 기본 라이브러리`

In [ ]:
import uuid
import warnings
from textwrap import dedent
from typing import Optional

warnings.filterwarnings("ignore")

from langchain_core.messages import HumanMessage
from langchain.agents import create_agent
from langchain_core.tools import tool

from langchain_community.vectorstores import FAISS
from langchain_tavily import TavilySearch
from langchain_upstage import UpstageEmbeddings

from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import MessagesState, StateGraph, START, END
from langgraph.prebuilt import ToolNode, tools_condition

## 2. Tool 정의

`(1) 사용자 정의 도구 - 메뉴 검색 (@tool)`
* 기존에 만들어 둔 메뉴 벡터저장소(FAISS)를 로드하고, 이를 검색하는 도구를 만듭니다.
* 도구의 **docstring 이 LLM 에게 전달되는 도구 설명**입니다. 언제 이 도구를 써야 하는지 분명히 적습니다.

In [ ]:
# 상수: 벡터DB 가 Upstage 임베딩으로 만들어져 있으므로 같은 모델을 사용해야 함
EMBEDDING_MODEL = "solar-embedding-1-large"
MENU_DB_PATH = "../db/menu_db"
MENU_SEARCH_K = 6  # 메뉴 검색 시 가져올 문서 수

embeddings_model = UpstageEmbeddings(model=EMBEDDING_MODEL)

# menu db 벡터 저장소 로드 (직접 만든 로컬 파일이므로 역직렬화 허용)
menu_db = FAISS.load_local(
    MENU_DB_PATH,
    embeddings_model,
    allow_dangerous_deserialization=True,
)

@tool
def search_menu(query: str) -> str:
    """레스토랑 메뉴(음식 종류, 재료, 가격 등)에 대한 정보를 메뉴 데이터베이스에서 검색합니다.
    Use this tool only for restaurant menu-related queries."""
    docs = menu_db.similarity_search(query, k=MENU_SEARCH_K)

    if not docs:
        return "관련 메뉴 정보를 찾을 수 없습니다."

    # 출처를 함께 전달해야 LLM 이 답변에 출처를 인용할 수 있음
    return "\n\n---\n\n".join(
        f'<Document source="{doc.metadata["source"]}"/>\n{doc.page_content}\n</Document>'
        for doc in docs
    )

`(2) LangChain 내장 도구 - 웹 검색 (Tavily)`
* 일반 웹 검색을 위해 `langchain_tavily.TavilySearch` 를 도구 안에서 사용합니다.
* (예전 `langchain_community` 의 `TavilySearchResults` 는 deprecated 입니다.)

In [ ]:
WEB_SEARCH_MAX_RESULTS = 3  # 웹 검색 결과 수

@tool
def search_web(query: str) -> str:
    """데이터베이스에 없는 정보나 최신 정보를 인터넷에서 검색합니다.
    Searches the internet for information that does not exist in the database or for the latest information."""
    tavily_search = TavilySearch(max_results=WEB_SEARCH_MAX_RESULTS)
    response = tavily_search.invoke({"query": query})  # {"query":..., "results": [{url, content, ...}]}
    results = response.get("results", [])

    if not results:
        return "관련 정보를 찾을 수 없습니다."

    return "\n\n---\n\n".join(
        f'<Document href="{item["url"]}"/>\n{item["content"]}\n</Document>'
        for item in results
    )

### 2-1. LLM 의 도구 선택 확인 (bind_tools)
* `bind_tools` 로 도구를 연결하면 LLM 은 질문에 맞는 도구와 인자를 `tool_calls` 로 돌려줍니다. (원리는 `03_Tool활용Basic` 참고)
* 질문에 따라 **메뉴 검색 / 웹 검색 / 도구 없음** 으로 갈리는지 확인합니다. 이 셀은 도구를 실제로 실행하지 않고 LLM 의 선택만 봅니다.

In [ ]:
tools = [search_menu, search_web]
llm_with_tools = llm.bind_tools(tools)  # 도구 설명이 포함된 LLM

test_questions = [
    "스테이크 메뉴의 가격은 얼마인가요?",                # 기대: search_menu
    "최근에 공개된 오픈소스 LLM 모델은 어떤 것들이 있나요?",  # 기대: search_web
    "3+3은 얼마인가요?",                              # 기대: 도구 없음
]

for question in test_questions:
    ai_message = llm_with_tools.invoke([HumanMessage(content=question)])
    print(f"질문: {question}")
    print("  tool_calls:", [(tc["name"], tc["args"]) for tc in ai_message.tool_calls] or "없음")

## 3. ReAct Agent
* ReAct(Reasoning and Acting): 가장 일반적인 에이전트 방식입니다.
    * 행동(act): 모델이 특정 도구를 호출
    * 관찰(observe): 도구의 출력을 모델에 다시 전달
    * 추론(reason): 모델이 도구 출력을 바탕으로 다음 행동을 결정 (또 다른 도구 호출 또는 최종 응답)
* 논문: https://arxiv.org/abs/2210.03629

`(1) create_agent 로 생성`
* 도구 목록과 `system_prompt` 만 전달하면 ReAct 루프가 담긴 그래프가 만들어집니다.
* `system_prompt` 는 에이전트 생성 시 한 번 지정하면 되고, 호출할 때는 `HumanMessage`(질문)만 전달합니다.
* 최신 모델은 도구를 **네이티브 tool calling** 으로 호출하므로, 프롬프트에 `Action:` / `Action Input:` 같은 텍스트 형식을 강제하지 않습니다. 사용 규칙과 출처 인용 형식만 지시합니다.

In [ ]:
# 시스템 프롬프트: 도구 사용 방침과 출처 인용 형식 지시
SYSTEM_PROMPT = dedent("""
당신은 사용자의 질문에 답하는 AI 어시스턴트입니다. 제공된 도구를 사용해 답변을 만드세요.

규칙:
1. 질문을 정확히 이해한 뒤, 필요한 정보는 도구로 조회합니다. (메뉴 정보는 search_menu, 최신/외부 정보는 search_web)
2. 도구가 필요 없는 질문(인사, 단순 계산 등)은 바로 답변합니다.
3. 도구 결과에 근거한 사실에는 반드시 출처를 아래 형식으로 인용합니다.
   [Source: 도구이름 | 문서 제목 또는 항목 | url 또는 파일 경로]
4. 도구 결과에 없는 내용은 지어내지 말고 모른다고 답합니다.
5. 답변은 명확하고 간결하게, 질문과 직접 관련된 내용 위주로 작성합니다.
""").strip()

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=SYSTEM_PROMPT,
)
print(type(agent))

In [ ]:
def ask(graph, query: str, config: Optional[dict] = None) -> None:
    """그래프에 질문 하나를 전달하고, 전체 메시지 흐름(Human → AI → Tool → AI)을 출력합니다."""
    result = graph.invoke({"messages": [HumanMessage(content=query)]}, config)
    for m in result["messages"]:
        m.pretty_print()


ask(agent, "스테이크 메뉴의 가격은 얼마인가요?")  # 메뉴 검색 도구 사용 기대

In [ ]:
ask(agent, "최근에 공개된 오픈소스 LLM 모델은 어떤 것들이 있나요?")  # 웹 검색 도구 사용 기대

In [ ]:
ask(agent, "안녕하세요?")  # 도구 없이 바로 응답하는지 확인

`(2) StateGraph + tools_condition 으로 직접 구성`
* `create_agent` 가 내부에서 하는 일을 직접 구성해 봅니다. (`should_continue` 를 직접 만드는 방식은 `03_Tool활용Basic` 참고)
* 라우팅 함수를 만들지 않고 LangGraph 제공 **`tools_condition`** 을 사용합니다.
    * 마지막 AI 메시지에 도구 호출이 있으면 → `"tools"` 노드로
    * 없으면 → `END` 로
* 주의: `tools_condition` 은 도구 실행 노드의 이름이 **`"tools"`** 라고 가정합니다.

메시지 흐름
* 도구를 사용하지 않는 경우: `SystemMessage` → `HumanMessage` → `AIMessage`(tool_calls 없음, 종료)
* 도구를 사용하는 경우: `SystemMessage` → `HumanMessage` → `AIMessage`(tool_calls) → `ToolMessage` → `AIMessage`(최종 답)

In [ ]:
from langchain_core.messages import SystemMessage


def call_model(state: MessagesState) -> MessagesState:
    """시스템 프롬프트를 맨 앞에 붙여 도구가 연결된 LLM 을 호출하고, 응답(답변 또는 tool_calls)을 상태에 추가합니다."""
    messages = [SystemMessage(content=SYSTEM_PROMPT)] + state["messages"]
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}


builder_tc = StateGraph(MessagesState)
builder_tc.add_node("agent", call_model)       # LLM 호출 노드
builder_tc.add_node("tools", ToolNode(tools))  # 도구 실행 노드 (이름은 반드시 "tools")

builder_tc.add_edge(START, "agent")
# 라우팅 함수 tools_condition: 도구 호출이 있으면 "tools", 없으면 END
builder_tc.add_conditional_edges("agent", tools_condition)
builder_tc.add_edge("tools", "agent")  # 도구 실행 후 다시 agent 로 (ReAct 루프)

my_graph_tc = builder_tc.compile()
print(type(my_graph_tc))

In [ ]:
# 그래프 구조 확인: mermaid 코드 출력 (https://mermaid.live/ 에 붙여 넣으면 그림으로 보임)
mermaid_code = my_graph_tc.get_graph().draw_mermaid()
print("Mermaid Code:")
print(mermaid_code)

In [ ]:
# 이미지로 직접 보려면: 
from IPython.display import Image, display
display(Image(my_graph_tc.get_graph().draw_mermaid_png()))

* https://mermaid.live/ 에서 `mermaid_code` 로 직접 확인할 수 있습니다. 점선(`-.->`)이 조건부 엣지입니다.
* 라우팅 함수 이름(`tools_condition`)은 그림에 나타나지 않습니다. 분기는 노드가 아니라 엣지의 조건이기 때문입니다.

In [ ]:
ask(my_graph_tc, "해산물 메뉴에는 어떤 것들이 있나요?")

## 4. MemorySaver

1. 상태의 일시성 문제
   * 기본적으로 그래프 실행 시 상태는 일시적(stateless)입니다.
   * 그래프를 다시 실행하면 상태가 초기화되어 다중 턴 대화가 어렵습니다.

2. MemorySaver 기능
   * 가장 쉽게 쓸 수 있는 체크포인터로, 각 단계 후 그래프 상태를 자동 저장합니다.
   * 그래프 상태를 위한 인메모리 키-값 저장소이며, 체크포인트부터 이어서 실행할 수 있습니다.
   * 프로세스가 끝나면 사라집니다. 운영 환경에서는 DB 기반 체크포인터(SQLite, Postgres 등)를 사용합니다.

3. 메모리의 필요성
   * 대화의 연속성: 여러 턴에 걸친 대화를 유지
   * 중단 허용: 대화 중 중단이 있어도 이전 상태를 복원
   * 유연한 상태 관리: 대화 스레드(`thread_id`)별로 독립 관리

### 4-1. 직접 구성한 그래프에 메모리 적용

In [ ]:
# 메모리 없이 실행: "이 중에"가 무엇을 가리키는지 알 수 없음 (이전 대화가 없음)
ask(my_graph_tc, "이 중에 하나만 추천해주세요.")

`MemorySaver 객체 사용`
* 그래프를 `compile` 할 때 `checkpointer` 로 `MemorySaver` 를 지정합니다.
* 주의: `compile()` 로 만든 그래프는 **새 객체**입니다. 위의 `my_graph_tc` 에는 메모리가 없습니다.

In [ ]:
memory = MemorySaver()  # 인메모리 체크포인터

# 같은 builder 로 체크포인터를 지정해 다시 컴파일
graph_memory = builder_tc.compile(checkpointer=memory)
print(type(graph_memory))

`체크포인터 사용`
* 호출할 때 `config` 에 `thread_id` 를 지정합니다. 같은 `thread_id` 는 같은 대화로 이어지고, 다른 `thread_id` 는 별개의 대화입니다.
* 체크포인터는 그래프의 각 단계에서 상태를 기록하며, `thread_id` 로 나중에 접근할 수 있습니다.

In [ ]:
config = {"configurable": {"thread_id": "1"}}
ask(graph_memory, "스테이크 메뉴의 가격은 얼마인가요?", config)

In [ ]:
# 같은 thread_id 로 후속 질문: "둘 중에"가 앞 답변의 메뉴들을 가리킴 (이전 메시지가 자동으로 이어짐)
ask(graph_memory, "둘 중에 더 저렴한 메뉴는 무엇인가요?", config)

In [ ]:
# 다른 thread_id 는 별개 대화: 앞 대화를 알지 못함
other_config = {"configurable": {"thread_id": "other"}}
ask(graph_memory, "둘 중에 더 저렴한 메뉴는 무엇인가요?", other_config)

# 저장된 상태 확인: thread_id 별 메시지 개수
print("thread 1    :", len(graph_memory.get_state(config).values["messages"]), "개 메시지")
print("thread other:", len(graph_memory.get_state(other_config).values["messages"]), "개 메시지")

### 4-2. create_agent 에 메모리 적용

In [ ]:
# create_agent 는 checkpointer 인자로 메모리를 바로 지정
memory = MemorySaver()

agent_memory = create_agent(
    model=llm,
    tools=tools,
    system_prompt=SYSTEM_PROMPT,
    checkpointer=memory,
)
print(type(agent_memory))

In [ ]:
config = {"configurable": {"thread_id": "2"}}
ask(agent_memory, "채식주의자를 위한 메뉴가 있나요?", config)

In [ ]:
# 같은 thread_id: "방금 답변"을 기억하는지 확인
ask(agent_memory, "방금 답변에 버섯이 포함된 메뉴가 있나요?", config)

## 5. Gradio 챗봇
* 메모리가 있는 에이전트(`agent_memory`)를 Gradio `ChatInterface` 에 연결합니다.
* Gradio 는 화면의 대화 기록(`history`)을 따로 넘기지만, 여기서는 **그래프의 체크포인터가 대화를 기억**하므로 `history` 는 쓰지 않고 새 메시지만 전달합니다.
* 브라우저 세션마다 `thread_id` 를 분리해야 사용자끼리 대화가 섞이지 않습니다. Gradio 의 `request.session_hash` 를 `thread_id` 로 사용합니다.
* 서버를 실행하면 노트북 셀이 계속 실행 상태가 됩니다. 마지막 셀의 `demo.close()` 로 종료하세요.

In [ ]:
import gradio as gr

# 예시 질문: Gradio UI 에 미리 표시되어 쉽게 테스트할 수 있음
EXAMPLE_QUESTIONS = [
    "채식주의자를 위한 메뉴를 추천해주세요.",
    "오늘의 스페셜 메뉴는 무엇인가요?",
    "파스타에 어울리는 음료는 무엇인가요?",
]


def chat_fn(message: str, history: list, request: gr.Request = None) -> str:
    """사용자 메시지를 에이전트에 전달하고 최종 답변 텍스트를 반환합니다.

    history: Gradio 가 넘기는 화면용 대화 기록 (메모리는 체크포인터가 관리하므로 사용하지 않음)
    request: 브라우저 세션 식별용. session_hash 를 thread_id 로 사용
    """
    thread_id = getattr(request, "session_hash", None) or str(uuid.uuid4())
    config = {"configurable": {"thread_id": thread_id}}

    try:
        result = agent_memory.invoke({"messages": [HumanMessage(content=message)]}, config)
        answer = result["messages"][-1].content  # 마지막 메시지 = 최종 AI 답변
        return answer if isinstance(answer, str) and answer else "응답을 생성하지 못했습니다."
    except Exception as e:
        # 내부 오류 내용은 서버 로그에만 남기고 사용자에게는 일반 메시지를 보여 줌
        print(f"Error occurred: {e}")
        return "죄송합니다. 응답을 생성하는 동안 오류가 발생했습니다. 다시 시도해 주세요."


demo = gr.ChatInterface(
    fn=chat_fn,
    title="레스토랑 메뉴 AI 어시스턴트",
    description="메뉴 정보, 추천, 음식 관련 질문에 답변해 드립니다. 정보의 출처를 함께 제공합니다.",
    examples=EXAMPLE_QUESTIONS,
)

# Gradio 애플리케이션 실행 (기본 주소: http://127.0.0.1:7860)
demo.launch()

In [ ]:
# 데모 종료 (실행 중인 Gradio 서버 포트 해제)
demo.close()